# DFCCNet — chạy train/test trên Kaggle GPU

Notebook này tự làm các việc sau (chi tiết xem `MANUAL.md`):

1. **Nạp source** — ưu tiên đọc thẳng từ path `/kaggle/input/<slug>/train.py`
   (đẩy source lên Kaggle Dataset), nếu không thì giải nén `code.zip` từ
   `/kaggle/input/<slug>/code.zip` về `/kaggle/working/src`.
2. **Nạp dataset** — tìm folder chứa `Train_Test_Val_chicken.json`, hoặc tự giải nén
   `Dense-Chicken.zip`. Nếu source/dataset đang mount từ `/kaggle/input` (read-only)
   thì sẽ **stage** sang `/kaggle/working` bằng symlink + copy.
3. Cài phần còn thiếu của `requirements.txt` (không cài lại `torch` vì Kaggle đã có bản CUDA).
4. Tự vá các lỗi đã biết của source (toggle `APPLY_FIXES`).
5. Chạy pipeline: `generate_conv.py` → `train.py` → `test.py` bằng `subprocess` + `cwd=REPO`.
6. Gom `stats.txt`, `*.pth`, `showimg/*.png` về `/kaggle/working/output`.

**Chuẩn bị trước khi chạy**

- Kernel → Accelerator → **GPU** (code gọi `.cuda()` cứng, không có nhánh CPU).
- Upload 1 Kaggle Dataset chứa source (folder có `train.py`, hoặc `code.zip`).
- Upload dataset: folder `chicken/` (có `images/`, `chicken_annocation.json`,
  `Train_Test_Val_chicken.json`, `chicken/conv/*.npy`) — hoặc để sẵn trong repo.
- Nếu muốn **train** thì cần `chicken/annocation/*.npy` (ground-truth density map);
  không có thì bật cell *"Generate density maps"* bên dưới.
- Đọc phần "Known issues" trong `MANUAL.md` trước khi để `APPLY_FIXES = True`.


In [ ]:
# ============================== CẤU HÌNH ==================================
import os, sys, json, shutil, zipfile, subprocess, time
from pathlib import Path

WORK  = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

# ---- các cờ lệnh (toggles) ------------------------------------------------
APPLY_FIXES        = True    # tự vá các lỗi đã biết của source (cell "Known fixes")
SKIP_TORCH_INSTALL = True    # True = chỉ cài phần thiếu, không đổi torch của Kaggle
REGEN_CONV         = False   # True = chạy lại generate_conv.py (cần chicken/single_chick/1..20.jpg)
GENERATE_DENSITY   = False   # True = sinh annocation/*.npy từ point annotation (fallback)
RUN_TRAIN          = True
RUN_TEST           = True

# ---- tham số chạy ---------------------------------------------------------
EPOCHS        = 50           # -ep
GPU           = 0            # -g
OUTPUT_DIR    = 'output'     # -o  (tương đối so với REPO)
SHOW_DIR      = 'showimg'    # -o  của test.py
TEST_SPLIT    = 'val'        # -ts : 'val' | 'test'
LEARNING_RATE = 1e-5         # -lr
EXTRA_LOSS_W  = 1e-9         # -wm
# ===========================================================================


In [ ]:
# ------------------------- 1. Nạp source (REPO) ---------------------------
def _repo_in(d: Path):
    if (d / 'train.py').exists():
        return d
    for sub in sorted(d.glob('*/train.py')):
        return sub.parent
    return None


def find_repo() -> Path:
    # (a) đọc source thẳng từ path : /kaggle/input/<slug>/train.py
    for d in sorted(p for p in INPUT.glob('*') if p.is_dir()):
        hit = _repo_in(d)
        if hit is not None:
            print('[REPO] đọc source từ dataset path :', hit)
            return hit
    # (b) upload code.zip : /kaggle/input/<slug>/code.zip
    zips = sorted(INPUT.glob('*/code.zip'))
    if not zips:
        zips = [z for z in sorted(INPUT.glob('*/*.zip'))
                if 'code' in z.stem.lower() or 'dfccnet' in z.stem.lower()]
    for z in zips:
        dest = WORK / 'src'
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        for cand in [dest, *sorted(dest.glob('*/'))]:
            if (cand / 'train.py').exists():
                print('[REPO] giải nén', z, '->', cand)
                return cand
    raise FileNotFoundError(
        'Không tìm thấy source. Đẩy repo lên Kaggle Dataset (folder có train.py) '
        'hoặc upload code.zip, rồi add dataset đó vào notebook này.'
    )


REPO = find_repo()

# /kaggle/input chỉ đọc -> copy về /kaggle/working để sửa code được
if str(REPO).startswith(str(INPUT)):
    dst = WORK / 'src'
    shutil.copytree(REPO, dst, dirs_exist_ok=True)
    print('[REPO] /kaggle/input read-only -> copy sang', dst)
    REPO = dst

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('[REPO] cwd =', Path.cwd())


In [ ]:
# ------------------------- 2. Nạp dataset (DATA) ---------------------------
def find_data() -> Path:
    # (a) folder đã giải nén, có file split
    for c in [REPO / 'chicken', *sorted(INPUT.glob('*/chicken')), *sorted(INPUT.glob('*'))]:
        if c.is_dir() and (c / 'Train_Test_Val_chicken.json').exists():
            print('[DATA] tìm thấy :', c)
            return c
    # (b) còn là zip -> giải nén
    for z in sorted(INPUT.glob('*/*.zip')):
        if 'dense' not in z.stem.lower() and 'chicken' not in z.stem.lower():
            continue
        dest = WORK / 'data_zip'
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        for cand in [dest, *sorted(dest.glob('*/'))]:
            if (cand / 'Train_Test_Val_chicken.json').exists():
                print('[DATA] giải nén', z, '->', cand)
                return cand
    raise FileNotFoundError(
        'Không tìm thấy dataset (cần file Train_Test_Val_chicken.json). '
        'Add dataset chứa chicken/ vào notebook này.'
    )


def writable_data(d: Path) -> Path:
    """Nếu dataset đang mount read-only thì stage sang /kaggle/working bằng symlink."""
    if not str(d).startswith(str(INPUT)):
        return d
    stage = WORK / 'data'
    stage.mkdir(parents=True, exist_ok=True)
    for name in ('images', 'single_chick', 'conv'):
        src, dst = d / name, stage / name
        if src.exists() and not dst.exists():
            dst.symlink_to(src, target_is_directory=src.is_dir())
    for f in d.glob('*.json'):
        if not (stage / f.name).exists():
            shutil.copyfile(f, stage / f.name)
    print('[DATA] read-only -> staged (ghi được) tại', stage)
    return stage


DATA = writable_data(find_data())

ANN_FILE = DATA / 'chicken_annocation.json'   # test.py / train.py (sau khi vá)
SPLIT    = DATA / 'Train_Test_Val_chicken.json'
IM_DIR   = DATA / 'images'
CONV_DIR = REPO / 'chicken' / 'conv'          # utils.py hardcode 'chicken/conv/...' (rel. cwd)

print(json.dumps({
    'REPO': str(REPO),
    'DATA': str(DATA),
    'annotation': ANN_FILE.exists(),
    'split': SPLIT.exists(),
    'n_images': len(list(IM_DIR.glob('*.jpg'))) if IM_DIR.is_dir() else 0,
    'conv_kernels': CONV_DIR.is_dir(),
}, indent=2, ensure_ascii=False))


In [ ]:
# ------------------------- 3. Cài dependency ------------------------------
import importlib.util

NEED = {'numpy': 'numpy', 'PIL': 'Pillow', 'cv2': 'opencv-python',
        'matplotlib': 'matplotlib', 'tqdm': 'tqdm'}

req = REPO / 'requirements.txt'
if req.exists() and not SKIP_TORCH_INSTALL:
    print('-> pip install -r requirements.txt')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
else:
    missing = [pip for mod, pip in NEED.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print('-> pip install', ' '.join(missing))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    else:
        print('-> đã đủ dependency, không cần cài gì')

import torch, torchvision
print('torch     ', torch.__version__, '| torchvision', torchvision.__version__)
print('cuda      ', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')
if not torch.cuda.is_available():
    raise RuntimeError('Code gọi .cuda() cứng -> Kernel > Accelerator > GPU trước khi chạy.')


In [ ]:
# ------------------------- 4. Known fixes + preflight ---------------------
def patch(rel: str, old: str, new: str, why: str):
    p = REPO / rel
    s = p.read_text(encoding='utf-8')
    if old not in s:
        print(f'[skip] {rel}: không tìm thấy pattern (có thể đã được sửa sẵn)')
        return
    p.write_text(s.replace(old, new, 1), encoding='utf-8')
    print(f'[fix ] {rel}: {why}')


if APPLY_FIXES:
    # (1) train.py dòng 22: `default=` thiếu giá trị -> SyntaxError (chặn mọi thứ)
    patch('train.py', 'default=, help', f'default={EPOCHS}, help',
          'sửa lỗi cú pháp default= của --epochs')

    # (2) tên file annotation lệch nhau: train.py mở 'chicken_annotation.json'
    #     nhưng file thật là 'chicken_annocation.json' (đúng như test.py vẫn đọc)
    patch('train.py', "'chicken_annotation.json'", "'chicken_annocation.json'",
          'đồng bộ tên file annotation với test.py')

    # (3) torch >= 2.6 đổi default weights_only=True -> không load được module object
    major, minor = (int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    if (major, minor) >= (2, 6):
        patch('test.py', 'torch.load("backbone.pth")',
              'torch.load("backbone.pth", weights_only=False)',
              'torch>=2.6: thêm weights_only=False')

    # (4) train.py chỉ os.mkdir (không tạo thư mục cha), test.py không tạo showimg/
    (REPO / OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    (REPO / SHOW_DIR).mkdir(parents=True, exist_ok=True)
    print(f'[fix ] tạo thư mục {OUTPUT_DIR}/ và {SHOW_DIR}/')

print('\n--- preflight ---')
CONV_NAMES = ['map3_conv', 'map4_conv'] + [
    f'{m}_conv_{s}' for m in ('map3', 'map4') for s in ('08', '09', '11', '12')]
checks = {
    'split json            ': SPLIT.exists(),
    'annotation json       ': ANN_FILE.exists(),
    'images/               ': IM_DIR.is_dir() and any(IM_DIR.glob('*.jpg')),
    'chicken/conv/*.npy    ': CONV_DIR.is_dir()
                              and all((CONV_DIR / f'{n}.npy').exists() for n in CONV_NAMES),
    'annocation/*.npy      ': (DATA / 'annocation').is_dir(),
    'single_chick/ (conv)  ': (DATA / 'single_chick').is_dir(),
    'result/*.pth          ': (REPO / 'result' / 'backbone.pth').exists(),
}
for k, v in checks.items():
    print(('  OK   ' if v else '  MISS ') + k)


### (Tuỳ chọn) Generate density maps

`train.py` đọc ground-truth density map tại `<data_path>annocation/<im_id>.npy`.
Thư mục này **không có trong repo** — thiếu thì train sẽ lỗi ngay epoch đầu.

Cell dưới đây sinh density map từ point annotation (`points`) bằng Gaussian kernel
ở **độ phân giải ảnh gốc** (đúng thứ mà `resizeImageWithGT` mong đợi).

> ⚠️ Đây là *fallback*: `SIGMA = 4.0` là giá trị mặc định phổ biến của các phương pháp
> density-map regression (MCNN…), **không phải tham số của paper DFCCNet**.
> Hãy kiểm tra lại (dùng `density_show.py`, và so tổng intensity với số point)
> trước khi tin kết quả.


In [ ]:
# ------------------------- 5. (Tuỳ chọn) sinh density map -----------------
if GENERATE_DENSITY:
    import numpy as np
    import cv2
    from PIL import Image

    gt_dir = DATA / 'annocation'
    gt_dir.mkdir(parents=True, exist_ok=True)
    anno = json.loads(ANN_FILE.read_text(encoding='utf-8'))

    SIGMA = 4.0
    k = int(6 * SIGMA) | 1
    g = cv2.getGaussianKernel(k, SIGMA)
    g2 = (g @ g.T).astype(np.float32)
    half = k // 2

    made = 0
    for im_id, a in anno.items():
        dst = gt_dir / (Path(im_id).stem + '.npy')
        if dst.exists():
            continue
        w, h = Image.open(IM_DIR / im_id).size
        dm = np.zeros((h, w), dtype=np.float32)
        pts = np.array(a.get('points', []), dtype=np.float32).reshape(-1, 2)
        for x, y in pts:
            xi, yi = int(round(float(x))), int(round(float(y)))
            x0, x1 = xi - half, xi + half + 1
            y0, y1 = yi - half, yi + half + 1
            gx0, gy0 = max(0, -x0), max(0, -y0)
            x0, y0 = max(0, x0), max(0, y0)
            x1, y1 = min(w, x1), min(h, y1)
            if x1 <= x0 or y1 <= y0:
                continue
            dm[y0:y1, x0:x1] += g2[gy0:gy0 + (y1 - y0), gx0:gx0 + (x1 - x0)]
        np.save(dst, dm)
        made += 1

    print(f'[density] sinh {made} file -> {gt_dir} (sigma={SIGMA})')
else:
    n = len(list((DATA / 'annocation').glob('*.npy'))) if (DATA / 'annocation').is_dir() else 0
    print(f'[density] GENERATE_DENSITY=False, hiện có {n} file .npy')


In [ ]:
# ------------------------- 6. Runner (subprocess) -------------------------
def run_script(script: str, args: dict, label: str, cwd: Path = REPO) -> None:
    """Chạy một script trong repo: in lệnh, đo thời gian, kiểm tra exit code."""
    cmd = [sys.executable, str(cwd / script)]
    for flag, value in args.items():
        cmd += [flag, str(value)]

    print('->', ' '.join(cmd), flush=True)
    started = time.perf_counter()
    return_code = subprocess.call(cmd, cwd=str(cwd), env=os.environ.copy())
    minutes = (time.perf_counter() - started) / 60
    print(f'-> {label}: exit={return_code} | {minutes:.1f} phút | cwd={cwd}')

    if return_code != 0:
        raise SystemExit(f'{label} thất bại (exit {return_code})')
    if 'torch' in sys.modules and torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ------------------------- 7. Feature convolution kernels ------------------
CONV_NAMES = ['map3_conv', 'map4_conv'] + [
    f'{m}_conv_{s}' for m in ('map3', 'map4') for s in ('08', '09', '11', '12')]
have_conv = CONV_DIR.is_dir() and all((CONV_DIR / f'{n}.npy').exists() for n in CONV_NAMES)

if REGEN_CONV or not have_conv:
    (REPO / 'chicken' / 'conv').mkdir(parents=True, exist_ok=True)
    # generate_conv.py đọc <data_path>single_chick/1.jpg .. 20.jpg
    run_script('generate_conv.py', {'-dp': str(DATA) + '/', '-g': GPU}, 'generate_conv')
else:
    print(f'[conv] đã có {len(CONV_NAMES)} kernel trong {CONV_DIR}, bỏ qua generate_conv.py')


In [ ]:
# ------------------------- 8. Train ---------------------------------------
if RUN_TRAIN:
    train_args = {
        '-dp': str(DATA) + '/',          # data_path (phải kết thúc bằng '/')
        '-o': str(REPO / OUTPUT_DIR),    # output_dir
        '-ts': TEST_SPLIT,               # split đánh giá sau mỗi epoch
        '-ep': EPOCHS,                   # epochs
        '-g': GPU,                       # gpu id
        '-wm': EXTRA_LOSS_W,             # extra_loss_weight
        '-lr': LEARNING_RATE,            # learning-rate
    }
    run_script('train.py', train_args, 'train')

    stats = REPO / OUTPUT_DIR / 'stats.txt'
    if stats.exists():
        print('\n--- stats.txt (loss,train_mae,train_rmse,val_mae,val_rmse) ---')
        print(stats.read_text(encoding='utf-8'))


In [ ]:
# ------------------------- 9. Test / evaluate -----------------------------
if RUN_TEST:
    # test.py load checkpoint TỪ CWD (repo root) -> copy về đây nếu chưa có
    for name in ('backbone.pth', 'generate_density.pth'):
        dst = REPO / name
        if dst.exists():
            continue
        for cand in (REPO / OUTPUT_DIR / name, REPO / 'result' / name):
            if cand.exists():
                shutil.copyfile(cand, dst)
                print(f'[ckpt] {cand} -> {dst}')
                break

    (REPO / SHOW_DIR).mkdir(parents=True, exist_ok=True)   # test.py không tự tạo

    run_script('test.py', {
        '-dp': str(DATA) + '/',
        '-ts': TEST_SPLIT,
        '-g': GPU,
        '-o': str(REPO / SHOW_DIR),
    }, 'test')


In [ ]:
# ------------------------- 10. Gom kết quả --------------------------------
OUT = WORK / 'output'
OUT.mkdir(parents=True, exist_ok=True)

for name in ('stats.txt', 'backbone.pth', 'generate_density.pth'):
    src = REPO / OUTPUT_DIR / name
    if src.exists():
        shutil.copyfile(src, OUT / name)

show = REPO / SHOW_DIR
if show.is_dir():
    shutil.copytree(show, OUT / 'showimg', dirs_exist_ok=True)

print('Đã gom kết quả vào', OUT)
for p in sorted(OUT.rglob('*')):
    print('  -', p.relative_to(OUT), f'({p.stat().st_size} bytes)' if p.is_file() else '')

print()
print('Để lưu lại vĩnh viễn: Save Version (Keep output only),')
print('hoặc "Add Output" / đẩy thư mục này lên một Kaggle Dataset mới.')
